# Short Guide: Processed Dataset

## a) Input Format
- **File type:** CSV (`raw_fire_data.csv`)
- **Raw columns included:**
  - `latitude`, `longitude` -> fire location coordinates
  - `frp` -> Fire Radiative Power (intensity)
  - `confidence` -> confidence score of detection
  - `satellite` -> Aqua/Terra (categorical)
  - `acq_date`, `acq_time` -> acquisition date and time

This is the raw dataset before preprocessing.

---

## b) Output Format
- **File type:** CSV (`cleaned_fire_data.csv`)
- **Final structure:** 74,029 rows × 7 columns
- **Columns included:**
  - `latitude` (float)
  - `longitude` (float)
  - `frp` (float, normalized using z‑score)
  - `confidence` (integer, target variable for modeling)
  - `satellite` (integer, encoded: Aqua = 0, Terra = 1)
  - `date` (YYYY‑MM‑DD)
  - `time` (HH:MM:SS)

This is the cleaned dataset, ready for modeling.

---

## c) Feature Descriptions
- **Latitude & Longitude** -> Fire location coordinates (numeric).
- **FRP (Fire Radiative Power)** -> Intensity of fire, normalized to mean = 0, std = 1.
- **Confidence** -> Detection confidence score (target variable for classification).
- **Satellite** -> Encoded categorical feature (Aqua = 0, Terra = 1).
- **Date** -> Calendar date of fire detection (used for daily aggregation).
- **Time** -> Time of fire detection (used for time‑of‑day analysis).


In [2]:
import pandas as pd

df = pd.read_csv("modis_2024_India.csv")


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74029 entries, 0 to 74028
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    74029 non-null  float64
 1   longitude   74029 non-null  float64
 2   brightness  74029 non-null  float64
 3   scan        74029 non-null  float64
 4   track       74029 non-null  float64
 5   acq_date    74029 non-null  str    
 6   acq_time    74029 non-null  int64  
 7   satellite   74029 non-null  str    
 8   instrument  74029 non-null  str    
 9   confidence  74029 non-null  int64  
 10  version     74029 non-null  float64
 11  bright_t31  74029 non-null  float64
 12  frp         74029 non-null  float64
 13  daynight    74029 non-null  str    
 14  type        74029 non-null  int64  
dtypes: float64(8), int64(3), str(4)
memory usage: 8.5 MB


## Observations
- Dataset has **74,029 rows** and **15 columns**.
- No **missing values** reported in `df.info()`.
- Contains both **numeric (float64, int64)** and **categorical (str)** features.
- Some columns are **metadata** (`instrument`, `version`, `type`, `daynight`) that may not be essential for modeling.



In [4]:
df.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,23.7690,86.3923,303.8,3.6,1.8,2024-01-01,346,Terra,MODIS,13,61.03,291.9,27.5,D,2
1,34.0996,74.9958,305.7,1.0,1.0,2024-01-01,522,Terra,MODIS,55,61.03,278.0,10.1,D,0
2,31.0805,78.2029,300.8,1.5,1.2,2024-01-01,522,Terra,MODIS,18,61.03,284.3,11.5,D,0
3,34.0908,74.9942,300.5,1.0,1.0,2024-01-01,522,Terra,MODIS,28,61.03,280.4,6.9,D,0
4,23.6291,74.3663,304.3,1.2,1.1,2024-01-01,524,Terra,MODIS,43,61.03,292.2,5.1,D,0


In [5]:
# Keep only the core features
df = df[['latitude','longitude','frp','confidence','satellite','acq_date','acq_time']]

# Check the new structure

print(df.head())


   latitude  longitude   frp  confidence satellite    acq_date  acq_time
0   23.7690    86.3923  27.5          13     Terra  2024-01-01       346
1   34.0996    74.9958  10.1          55     Terra  2024-01-01       522
2   31.0805    78.2029  11.5          18     Terra  2024-01-01       522
3   34.0908    74.9942   6.9          28     Terra  2024-01-01       522
4   23.6291    74.3663   5.1          43     Terra  2024-01-01       524



- Kept **latitude, longitude, frp, confidence, satellite, acq_date, acq_time** as core features.
- Dropped **instrument, version, type, daynight, brightness, bright_t31, scan, track** because:
  - They are either **constant values** (instrument, version),
  - **Redundant with FRP** (brightness, bright_t31),
  - **Metadata not essential** for prototype (scan, track, type, daynight).
- Inspired by EDA: focusing on **location, fire intensity, reliability, and time** ensures a lean dataset for modeling.


In [6]:
# Combine acq_date and acq_time into a single datetime column
df['datetime'] = pd.to_datetime(df['acq_date'] + ' ' + df['acq_time'].astype(str).str.zfill(4),
                                 format='%Y-%m-%d %H%M')

# Drop the old columns since we now have a unified datetime
df = df.drop(columns=['acq_date', 'acq_time'])

# Check the new structure
print(df.info())



<class 'pandas.DataFrame'>
RangeIndex: 74029 entries, 0 to 74028
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   latitude    74029 non-null  float64       
 1   longitude   74029 non-null  float64       
 2   frp         74029 non-null  float64       
 3   confidence  74029 non-null  int64         
 4   satellite   74029 non-null  str           
 5   datetime    74029 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 3.4 MB
None


## Datetime Transformation
- Combined **acq_date** and **acq_time** into a single **datetime** column.
- Dropped the original date and time columns to avoid redundancy.
- Inspired by EDA: a unified datetime makes **time-based analysis** and **model integration** smoother.


In [7]:
# Encode satellite column: Aqua -> 0, Terra -> 1
df['satellite'] = df['satellite'].map({'Aqua': 0, 'Terra': 1})

# Check the transformation
print(df['satellite'].unique())



[1 0]


In [8]:
print(df.head())


   latitude  longitude   frp  confidence  satellite            datetime
0   23.7690    86.3923  27.5          13          1 2024-01-01 03:46:00
1   34.0996    74.9958  10.1          55          1 2024-01-01 05:22:00
2   31.0805    78.2029  11.5          18          1 2024-01-01 05:22:00
3   34.0908    74.9942   6.9          28          1 2024-01-01 05:22:00
4   23.6291    74.3663   5.1          43          1 2024-01-01 05:24:00


In [9]:
# Split datetime into separate date and time columns
df['date'] = df['datetime'].dt.date
df['time'] = df['datetime'].dt.time

# Drop the original datetime column
df = df.drop(columns=['datetime'])

print(df.head())



   latitude  longitude   frp  confidence  satellite        date      time
0   23.7690    86.3923  27.5          13          1  2024-01-01  03:46:00
1   34.0996    74.9958  10.1          55          1  2024-01-01  05:22:00
2   31.0805    78.2029  11.5          18          1  2024-01-01  05:22:00
3   34.0908    74.9942   6.9          28          1  2024-01-01  05:22:00
4   23.6291    74.3663   5.1          43          1  2024-01-01  05:24:00


In [10]:
df['frp'].describe()

count    74029.000000
mean        24.841774
std         73.853068
min          0.000000
25%          8.000000
50%         12.400000
75%         22.200000
max       7528.500000
Name: frp, dtype: float64

## FRP Normalization
- Applied **z-score normalization** to the FRP column: `(value - mean) / std`.
- Inspired by modeling needs: FRP values vary widely, normalization ensures **balanced scale** across features.
- Helps the model avoid bias toward large FRP values and improves training stability.


In [11]:
# Apply z-score normalization to FRP
df['frp'] = (df['frp'] - df['frp'].mean()) / df['frp'].std()

# Check the transformation
print(df['frp'].describe())


count    7.402900e+04
mean    -5.528544e-17
std      1.000000e+00
min     -3.363675e-01
25%     -2.280443e-01
50%     -1.684666e-01
75%     -3.577068e-02
max      1.016025e+02
Name: frp, dtype: float64


## FRP Normalization Insights
- Before: FRP mean **24.84**, std **73.85**, max **7528.5** -> highly skewed with extreme outliers.
- After: FRP mean ≈ **0**, std = **1**, max ≈ **101.6** -> standardized scale, centered distribution.
- Most values lie slightly below mean (25%–75% between **-0.228 and -0.035**).
- Outliers remain but expressed as **standard deviations** -> easier to interpret and model.


In [12]:
df

,latitude,longitude,frp,confidence,satellite,date,time
0,23.7690,86.3923,0.035993,13,1,2024-01-01,03:46:00
1,34.0996,74.9958,-0.199610,55,1,2024-01-01,05:22:00
2,31.0805,78.2029,-0.180653,18,1,2024-01-01,05:22:00
3,34.0908,74.9942,-0.242939,28,1,2024-01-01,05:22:00
4,23.6291,74.3663,-0.267311,43,1,2024-01-01,05:24:00
...,...,...,...,...,...,...,...
74024,22.0444,83.7350,-0.011398,100,1,2024-12-31,15:44:00
74025,23.5604,87.2335,-0.214504,46,1,2024-12-31,15:45:00
74026,23.6847,86.3956,-0.218566,48,1,2024-12-31,15:45:00
74027,23.6663,86.9220,-0.222628,37,1,2024-12-31,15:45:00


In [13]:
# Save the cleaned dataset to a new CSV file
df.to_csv("Processed_fire_data.csv", index=False)



## Final Output
- Saved cleaned dataset as **Processed_fire_data.csv**.
- Final structure: 74,029 rows × 7 columns -> latitude, longitude, frp (normalized), confidence, satellite (encoded), date, time.
- Ready for modeling: lean, standardized, and reproducible dataset.


In [14]:
import pandas as pd

def load_data(filepath):
    """
    Load raw fire dataset from CSV.
    Args:
        filepath (str): Path to the CSV file.
    Returns:
        pd.DataFrame: Loaded dataset.
    """
    return pd.read_csv(filepath)


## Function: load_data()
- Purpose: Load the raw fire dataset from a CSV file.
- Input: Filepath (string) -> location of the raw dataset.
- Output: Pandas DataFrame containing the raw data.
- Why: Encapsulates the loading step into a reusable function, ensuring consistency and avoiding repeated code.


In [15]:
def preprocess_data(df):
    """
    Preprocess fire dataset:
    - Select relevant columns
    - Transform datetime into date & time
    - Encode satellite
    - Normalize FRP
    Args:
        df (pd.DataFrame): Raw dataset
    Returns:
        pd.DataFrame: Cleaned dataset
    """
    # Column selection
    df = df[['latitude','longitude','frp','confidence','satellite','acq_date','acq_time']]

    # Datetime transformation
    df['datetime'] = pd.to_datetime(df['acq_date'] + ' ' + df['acq_time'].astype(str).str.zfill(4),
                                    format='%Y-%m-%d %H%M')
    df['date'] = df['datetime'].dt.date
    df['time'] = df['datetime'].dt.time
    df = df.drop(columns=['acq_date','acq_time','datetime'])

    # Encoding satellite
    df['satellite'] = df['satellite'].str.strip().str.capitalize()
    df['satellite'] = df['satellite'].map({'Aqua':0, 'Terra':1})

    # FRP normalization
    df['frp'] = (df['frp'] - df['frp'].mean()) / df['frp'].std()

    return df


## Function: preprocess_data()
- Purpose: Apply preprocessing pipeline to the raw dataset.
- Steps included:
  1. **Column Selection** -> Keep latitude, longitude, frp, confidence, satellite, acq_date, acq_time.
  2. **Datetime Transformation** -> Combine acq_date + acq_time -> split into date & time columns.
  3. **Encoding** -> Convert satellite (Aqua, Terra) into numeric (0, 1).
  4. **FRP Normalization** -> Apply z-score normalization `(value - mean) / std`.
- Input: Raw DataFrame.
- Output: Cleaned DataFrame with 7 columns -> latitude, longitude, frp (normalized), confidence, satellite (encoded), date, time.
- Why: Makes preprocessing reproducible, modular, and ready for modeling.

In [15]:
# Load raw dataset and preprocess in one step
dataf = preprocess_data(load_data("modis_2024_India.csv"))

# Confirm the cleaned dataset
print(dataf.shape)
print(dataf.head())


## Model Requirements Check


In [16]:
# 1. Check data types
print(df.dtypes)


latitude      float64
longitude     float64
frp           float64
confidence      int64
satellite       int64
date           object
time           object
dtype: object


In [19]:
df['date'] = pd.to_datetime(df['date'])
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.time
df['hour'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.hour


In [20]:
df.head()

,latitude,longitude,frp,confidence,satellite,date,time,hour
0,23.7690,86.3923,0.035993,13,1,2024-01-01,03:46:00,3
1,34.0996,74.9958,-0.199610,55,1,2024-01-01,05:22:00,5
2,31.0805,78.2029,-0.180653,18,1,2024-01-01,05:22:00,5
3,34.0908,74.9942,-0.242939,28,1,2024-01-01,05:22:00,5
4,23.6291,74.3663,-0.267311,43,1,2024-01-01,05:24:00,5


In [21]:
df.dtypes

latitude            float64
longitude           float64
frp                 float64
confidence            int64
satellite             int64
date          datetime64[s]
time                 object
hour                  int32
dtype: object


-  All core features (`latitude`, `longitude`, `frp`, `confidence`, `satellite`) are numeric.
- `date` is now converted to `datetime64[s]` -> usable for extracting year, month, day.
- `time` remains `object` -> extracted `hour` as `int32` for modeling.
-  Dataset is now model‑ready: no string/object columns remain in features, only numeric or datetime.


In [17]:
# 2. Check for missing values
print(df.isnull().sum())

latitude      0
longitude     0
frp           0
confidence    0
satellite     0
date          0
time          0
dtype: int64


# Null Values Check – Notes

- No missing values detected in any column.
- Dataset is complete and ready for modeling without imputation or dropping rows.


In [18]:
# 3. Check target class balance
print(df['confidence'].value_counts())

confidence
67    2044
66    2027
65    1979
63    1963
64    1961
      ... 
4       11
6        8
2        4
3        3
1        3
Name: count, Length: 101, dtype: int64


## Target Class Balance – Notes

- Confidence values are unevenly distributed.
- Majority classes: 63–67 with around 2000 samples each.
- Minority classes: 2–6 with fewer than 15 samples each.
- The dataset is imbalanced, so the model may favor majority classes.
- Balancing methods such as resampling or class weighting may be needed.



## Required Libraries for Scaling & Encoding

- pandas, numpy (already used for preprocessing)
- scikit-learn -> scaling, encoding, train/test split, metrics
- imbalanced-learn -> resampling for imbalanced datasets
- great-expectations -> data validation
- ydata-profiling -> dataset profiling reports


In [22]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['frp_minmax'] = scaler.fit_transform(df[['frp']])

# Quick check
print(df[['frp', 'frp_minmax']].head())


        frp  frp_minmax
0  0.035993    0.003653
1 -0.199610    0.001342
2 -0.180653    0.001528
3 -0.242939    0.000917
4 -0.267311    0.000677


## Min-Max Scaling – Insights

- FRP values have been rescaled into the range [0,1].
- Original FRP values vary widely, while z-score normalization centers them around mean 0 with both positive and negative values.
- Min-Max scaling compresses all values into a uniform scale, making them directly comparable across features.
- This is especially useful for distance-based models (KNN, SVM, Neural Networks) where magnitude differences affect performance.
- Both z-score and Min-Max versions are retained in the dataset to provide flexibility for different modeling approaches.


In [25]:
from sklearn.model_selection import train_test_split

# Features (X) and target (y)
X = df.drop('confidence', axis=1)   # all columns except target
y = df['confidence']                # target column

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify = y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)


Training set size: (59223, 8)
Testing set size: (14806, 8)


## Train/Test Split – Explanation

- Features (X) = all columns except confidence.
- Target (y) = confidence column.
- Dataset split into 80% training and 20% testing.
- Training set used for model learning.
- Testing set used for evaluation on unseen data.
- Random state ensures reproducibility of the split.


In [26]:
print("Train class distribution:\n", y_train.value_counts())
print("Test class distribution:\n", y_test.value_counts())


Train class distribution:
 confidence
67    1635
66    1622
65    1583
63    1570
64    1569
      ... 
4        9
6        6
2        3
3        3
1        3
Name: count, Length: 101, dtype: int64
Test class distribution:
 confidence
67    409
66    405
65    396
63    393
64    392
     ... 
7       3
6       2
4       2
5       2
2       1
Name: count, Length: 99, dtype: int64


## Train/Test Split – Class Distribution

- Majority confidence values (63–67) are consistently represented in both train and test sets.
- Minority confidence values (2–6) remain very underrepresented in both sets.
- The imbalance is preserved after splitting.
- Stratified splitting can be used to maintain proportional representation of classes in train and test sets.
